# 03 — Gene Scoring with ULM

Score cells using the consensus disease gene list from notebook 02 via
`scutils.tl.compute_ulm_scores` (Univariate Linear Model).

**Workflow:**
1. Load AnnData and consensus gene list
2. Score cells with ULM
3. Visualise scores on UMAP embeddings
4. Compare score distributions across conditions and cell types

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `ADATA_PATH` — path to AnnData (can be the one saved by notebook 01)
- `CONSENSUS_GENES_PATH` — path to the consensus gene list from notebook 02
- `SCORE_NAME` — column name for the ULM score in `adata.obs`
- `LAYER` — expression layer to use (`None` = `adata.X`)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
ADATA_PATH          = Path("results/disease_subclusters/dataset_A/adata_with_subclusters.h5ad")
CONSENSUS_GENES_PATH = Path("results/disease_subclusters/consensus/consensus_genes.txt")

# ── Column names in adata.obs ─────────────────────────────────────────────────
DISEASE_KEY  = "disease"
CELLTYPE_KEY = "cell_type"
RESULT_KEY   = "disease_subcluster"   # from notebook 01

# ── ULM scoring settings ──────────────────────────────────────────────────────
SCORE_NAME   = "disease_consensus_ulm"
SET_NAME     = "disease_consensus"
LAYER: str | None = None    # expression layer; None = adata.X
MIN_N        = 5            # minimum genes required for scoring
RETURN_PVALS = True

# ── Plotting ──────────────────────────────────────────────────────────────────
SAVE_FIGS = True

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("results/disease_subclusters/gene_scoring")

print("Configuration loaded.")
print(f"  AnnData path     : {ADATA_PATH}")
print(f"  Consensus genes  : {CONSENSUS_GENES_PATH}")
print(f"  Score name       : {SCORE_NAME}")
print(f"  Layer            : {LAYER!r}")
print(f"  Output dir       : {OUTPUT_DIR}")

## 3. Load Data & Gene List

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)

# Load consensus genes
consensus_genes = [
    g.strip() for g in CONSENSUS_GENES_PATH.read_text().strip().split("\n") if g.strip()
]
print(f"\nLoaded {len(consensus_genes)} consensus genes.")

# Check overlap with adata
found = [g for g in consensus_genes if g in adata.var_names]
print(f"Found in adata.var_names: {len(found)} / {len(consensus_genes)}")

## 4. Compute ULM Scores

In [ ]:
scutils.tl.compute_ulm_scores(
    adata,
    genes=consensus_genes,
    set_name=SET_NAME,
    score_name=SCORE_NAME,
    layer=LAYER,
    min_n=MIN_N,
    return_pvals=RETURN_PVALS,
)

print(f"\n✓ ULM scores stored in adata.obs['{SCORE_NAME}']")
print(adata.obs[SCORE_NAME].describe())

if RETURN_PVALS:
    pval_col = f"{SCORE_NAME}_pval"
    n_sig = (adata.obs[pval_col] < 0.05).sum()
    print(f"\nCells with significant score (p < 0.05): {n_sig:,} / {adata.n_obs:,}")

## 5. Visualise Scores on UMAP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Compute symmetric colour limits
import numpy as np
score_vals = adata.obs[SCORE_NAME].dropna()
vmax = np.percentile(np.abs(score_vals), 99.9)

# Score heatmap
sc.pl.umap(
    adata,
    color=SCORE_NAME,
    ax=axes[0],
    show=False,
    title=f"ULM score: {SET_NAME}",
    cmap="bwr",
    vmin=-vmax,
    vmax=vmax,
)

# Disease labels for context
sc.pl.umap(
    adata,
    color=DISEASE_KEY,
    ax=axes[1],
    show=False,
    title=f"Disease ({DISEASE_KEY})",
)

fig.tight_layout()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_FIGS:
    fig_path = OUTPUT_DIR / "umap_ulm_score.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 6. Score Distribution by Disease

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    data=adata.obs,
    x=DISEASE_KEY,
    y=SCORE_NAME,
    ax=ax,
    inner="box",
    cut=0,
)
ax.set_title(f"ULM score distribution by disease")
ax.set_ylabel(SCORE_NAME)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

if SAVE_FIGS:
    fig_path = OUTPUT_DIR / "violin_score_by_disease.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 7. Score Distribution by Cell Type × Disease

In [ ]:
celltypes = adata.obs[CELLTYPE_KEY].unique().tolist()
n_ct = len(celltypes)
ncols = min(3, n_ct)
nrows = math.ceil(n_ct / ncols) if n_ct > 0 else 1

import math

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(5 * ncols, 4 * nrows),
    squeeze=False,
)

for idx, ct in enumerate(sorted(celltypes)):
    row, col = divmod(idx, ncols)
    ax = axes[row][col]

    ct_mask = adata.obs[CELLTYPE_KEY] == ct
    sns.violinplot(
        data=adata.obs[ct_mask],
        x=DISEASE_KEY,
        y=SCORE_NAME,
        ax=ax,
        inner="box",
        cut=0,
    )
    ax.set_title(ct)
    ax.set_ylabel("" if col > 0 else SCORE_NAME)
    ax.tick_params(axis="x", rotation=45)

# Hide empty axes
for idx in range(n_ct, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row][col].set_visible(False)

fig.suptitle("ULM score by cell type × disease", fontsize=13, y=1.01)
fig.tight_layout()

if SAVE_FIGS:
    fig_path = OUTPUT_DIR / "violin_score_by_celltype_disease.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 8. Score Significance Heatmap

Per-sample aggregated ULM score across cell types and disease conditions
with statistical significance annotations.

In [ ]:
# Requires a sample-level column for aggregation — reuse PSEUDOBULK_GROUP_KEY
# if available, otherwise set SAMPLE_COL here.
SAMPLE_COL = "sampleID"          # column in adata.obs identifying biological samples
CONTROL_LABEL = "Control"        # reference level in DISEASE_KEY for p-value comparisons

fig = scutils.pl.heatmap_feature_aggregated_three_categories(
    adata,
    feature=SCORE_NAME,
    x=DISEASE_KEY,
    y=CELLTYPE_KEY,
    panel=CELLTYPE_KEY,         # one panel per cell type (single-panel if desired)
    sample_col=SAMPLE_COL,
    x_ref=CONTROL_LABEL,
    agg_fn="mean",
    min_cells=10,
    cmap="bwr",
    zero_center=True,
    show_stats=True,
    show_ns=False,
    ncols=3,
    figsize=(4, 3),
)

if SAVE_FIGS:
    fig_path = OUTPUT_DIR / "heatmap_score_significance.png"
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Saved: {fig_path}")

plt.show()
plt.close(fig)

## 9. Save Scored AnnData

In [ ]:
scored_path = OUTPUT_DIR / "adata_scored.h5ad"
adata.write_h5ad(scored_path)
print(f"Saved scored AnnData: {scored_path}")
print(f"\n✓ All outputs saved to {OUTPUT_DIR}")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Load data | `sc.read_h5ad()` | `ADATA_PATH` |
| Load genes | `Path.read_text()` | `CONSENSUS_GENES_PATH` |
| ULM scoring | `scutils.tl.compute_ulm_scores()` | `SCORE_NAME`, `LAYER`, `MIN_N` |
| UMAP visualisation | `sc.pl.umap()` | `SCORE_NAME`, `DISEASE_KEY` |
| Violin plots | `sns.violinplot()` | `DISEASE_KEY`, `CELLTYPE_KEY` |
| Export | `adata.write_h5ad()` | `OUTPUT_DIR` |